In [4]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import time
import json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import gensim.downloader as api

In [11]:
%load_ext autoreload
%autoreload 2

from test import *
from Data.Tool import *
from Model.Tool import *
from Model.model import *

VECTOR_SIZE  = 300     # dimension des embeddings FastText
HIDDEN_DIM   = 128
BIDIRECTIONAL= True
DROPOUT      = 0.3
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 1e-3
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED         = 42
MAX_LEN      = 26

torch.manual_seed(SEED)
np.random.seed(SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
df = load_and_prepare(
        corpus_path = "Data/train/corpus.tache1.learn.utf8",
        pkl_path    = "Data/train/sequences_fasttext_fr.pkl",
        vector_size = 300,
    )
print(f"Dataset chargé : {len(df)} lignes | Labels : {df['Label'].value_counts().to_dict()}")

train_df, val_df = train_test_split( df, test_size=0.2, random_state=SEED, stratify=df["Label"])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(val_df)}")

📂 Chargement du corpus...
   → 57413 lignes chargées
🧹 Nettoyage du texte...
📦 Chargement des séquences depuis 'Data/train/sequences_fasttext_fr.pkl'...
   → 57413 séquences fusionnées
🗑️  Suppression des lignes vides...
   → 35 lignes supprimées | 57378 lignes restantes
🔍 Vérification cohérence tokens ↔ vecteurs...


Vérification: 100%|██████████| 57378/57378 [00:01<00:00, 33034.79it/s]



✅ Lignes finales       : 57378
✅ Balance C/M         : {'C': 49855, 'M': 7523}
✅ Erreurs restantes  : 0
🎉 Pipeline prêt pour l'entraînement !
Dataset chargé : 57378 lignes | Labels : {'C': 49855, 'M': 7523}
Train : 45902 | Val : 11476 | Test : 11476


In [9]:
class RNN(nn.Module):
    def __init__(self, vector_size, hidden_dim, bi=False, dropout=0.2):
        super().__init__()
        self.rnn = nn.GRU(
            input_size=vector_size,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bi,
            num_layers=2,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * (1 + bi), 1)

    def forward(self, x):
        rnn_out, _ = self.rnn(x)
        pooled = torch.max(rnn_out, dim=1)[0]
        return self.fc(self.dropout(pooled)).squeeze(-1)


In [ ]:
# ── Hyperparamètres ──────────────────────────────────────────
VECTOR_SIZE   = 300
HIDDEN_DIM    = 128
BIDIRECTIONAL = True
DROPOUT       = 0.3
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 3e-4
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED          = 42
torch.manual_seed(SEED)

# ── Splits ───────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["Label"])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df["Label"])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

# ── Dataloaders ──────────────────────────────────────────────
train_loader = DataLoader(SentenceDatasetContext(train_df, max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentenceDatasetContext(val_df,   max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SentenceDatasetContext(test_df,  max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

# ── Modèle ───────────────────────────────────────────────────
model     = RNN(VECTOR_SIZE, HIDDEN_DIM, bi=BIDIRECTIONAL, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

n_C = (train_df["Label"] == "C").sum()
n_M = (train_df["Label"] == "M").sum()
pos_weight = torch.tensor([n_M / n_C], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Boucle d'entraînement ────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    # -- Train
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss    += loss.item() * len(y)
        tr_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
        tr_total   += len(y)

    # -- Val
    model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y   = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            vl_loss    += loss.item() * len(y)
            vl_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
            vl_total   += len(y)

    tr_loss /= tr_total
    vl_loss /= vl_total
    scheduler.step(vl_loss)

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss={tr_loss:.4f} Acc={tr_correct/tr_total:.4f} | Val Loss={vl_loss:.4f} Acc={vl_correct/vl_total:.4f}")


# ── Evaluation finale sur test ───────────────────────────────
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X      = X.to(DEVICE)
        logits = model(X)
        preds  = (torch.sigmoid(logits) > 0.5).float().cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

label_map  = {1.0: "C", 0.0: "M"}
pred_names = [label_map[p] for p in all_preds]
true_names = [label_map[l] for l in all_labels]

print(classification_report(true_names, pred_names, target_names=["M", "C"]))

Train : 43893 | Val : 4878 | Test : 8607


/Users/isaac/Library/Python/3.9/lib/python/site-packages/torch/optim/lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 01/20 | Train Loss=0.1614 Acc=0.6333 | Val Loss=0.1661 Acc=0.8188
Epoch 02/20 | Train Loss=0.1399 Acc=0.7606 | Val Loss=0.1519 Acc=0.7776


KeyboardInterrupt: 